# ECC/EMC â€” Kaggle Training: EmoSet Dataset

Supervised (Table 4) + Semi-supervised (Section 4.2.1) on EmoSet-118K.

## Before you run
1. **Enable GPU:** Settings â†’ Session options â†’ GPU (P100 or T4). Phone verification required.
2. **Add datasets:** Add the public EmoSet-118K dataset (https://www.kaggle.com/datasets/georgeliil/emoset) in the sidebar.
3. **Persistent checkpoints (recommended):** Add a private Kaggle Dataset for checkpoints, e.g. `your-username/ecc-checkpoints`. Mount it and set `PERSISTENT_DIR` below.
4. **Add GitHub token secret:** Kaggle Settings â†’ Secrets â†’ `GITHUB_TOKEN` = your GitHub PAT (repo scope).
5. **Edit `EMOSET_DATASET_SLUG`** below.

## Why persistent checkpoints?
EmoSet has ~82K+ images; a full 80-epoch ResNet50 run likely exceeds Kaggle's 12-hour session
cap. With `PERSISTENT_DIR` set, `checkpoint.pt` and `best.pt` are saved to a mounted Kaggle
Dataset every epoch, so training resumes across sessions.

## Results
All outputs go to `/kaggle/working/runs/emoset_*` â€” separate from FI results.

In [ ]:
# ===================== CONFIGURE THESE =====================
# Public EmoSet-118K dataset
EMOSET_DATASET_SLUG = "georgeliil/emoset"   # <-- REPLACE if different

# Persistent checkpoint Dataset (private Kaggle Dataset, e.g. your-username/ecc-checkpoints)
# Leave empty to disable persistent checkpoints (not recommended for EmoSet)
PERSISTENT_DIR = ""   # e.g. "/kaggle/input/ecc-checkpoints"

# Which experiments to run
RUN_SUPERVISED = True
RUN_SSL = True
# ==========================================================

print("EmoSet dataset slug:", EMOSET_DATASET_SLUG)
print("Persistent dir:", PERSISTENT_DIR or "(disabled)")

In [ ]:
# Install dependencies
!pip -q install torch torchvision numpy pyyaml pillow

In [ ]:
# Clone repo using GitHub token secret (added in Kaggle Settings -> Secrets)
import os
import subprocess

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_token = user_secrets.get_secret("GITHUB_TOKEN")
    repo_url = f"https://{secret_token}@github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"
except Exception:
    # Fallback to public URL if no secret is set
    repo_url = "https://github.com/Rahul-Sai-Indeevar/To-Err-Like-Human.git"

dest = "/kaggle/working/ecc-ver"

# Clone the repository if it doesn't already exist
if not os.path.exists(dest):
    subprocess.run(["git", "clone", repo_url, dest], check=True)
    print("Cloned repo to", dest)
else:
    print("Repository already exists at", dest)

# Change directory and install/add to path if needed
os.chdir(dest)
print("Current dir:", os.getcwd())


In [ ]:
# Install ecc-ver package
!pip -q install -e .

In [ ]:
# Symlink EmoSet dataset into data/ layout
import os

os.makedirs("data", exist_ok=True)

# Robustly find EmoSet/ anywhere under /kaggle/input
found = False
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth > 5:
        dirs[:] = []
        continue
    for d in dirs:
        if d in ("EmoSet", "EmoSet_splits", "image"):
            src = os.path.join(root, d)
            dst = f"data/{d}"
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"Linked {d} -> {dst} (from {src})")
            found = True

if not found:
    print("WARNING: Could not find EmoSet/EmoSet_splits/image under /kaggle/input")
    print("Available under /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth <= 3:
            indent = " " * 4 * depth
            print(f"{indent}{os.path.basename(root)}/ --> subdirs: {dirs}")

print("\nData layout:")
!find data -maxdepth 3 | head -30

In [ ]:
# ===================== GENERATE EMOSET SPLITS (if needed) =====================
import os
from pathlib import Path
import subprocess

splits_dir = Path("data/EmoSet_splits")
image_dir = Path("data/image")

if not (splits_dir / "classes.txt").exists():
    print("Generating EmoSet train/val/test splits from data/image...")
    if not image_dir.exists():
        print(f"ERROR: {image_dir} not found. Make sure EmoSet dataset is mounted.")
    else:
        result = subprocess.run(
            ["ecc-splits", "--root", str(image_dir), "--out", str(splits_dir),
             "--val", "0.05", "--test", "0.15", "--seed", "0"],
            capture_output=True, text=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print(f"ERROR: {result.stderr}")

print("\nEmoSet data layout:")
!find data -maxdepth 3 | head -50


In [ ]:
# Quick smoke test (CPU, no data needed)
!python -m pytest -q

In [ ]:
# ===================== SUPERVISED (EmoSet, ResNet50, Table 4) =====================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if RUN_SUPERVISED:
    persistent_arg = f"--set persistent_dir={PERSISTENT_DIR}" if PERSISTENT_DIR else ""
    
    # alpha = 0 (pure CE)
    print("=== alpha=0 ===")
    cmd = f"python -m ecc.supervised --config configs/emoset_resnet50.yaml --set loss.alpha=0.0 output_dir=/kaggle/working/runs/emoset_r50_a0 {persistent_arg}"
    print("Running:", cmd)
    !{cmd}

    # alpha = 0.2
    print("=== alpha=0.2 ===")
    cmd = f"python -m ecc.supervised --config configs/emoset_resnet50.yaml --set loss.alpha=0.2 output_dir=/kaggle/working/runs/emoset_r50_a0.2 {persistent_arg}"
    print("Running:", cmd)
    !{cmd}

    # alpha = 1.0
    print("=== alpha=1.0 ===")
    cmd = f"python -m ecc.supervised --config configs/emoset_resnet50.yaml --set loss.alpha=1.0 output_dir=/kaggle/working/runs/emoset_r50_a1.0 {persistent_arg}"
    print("Running:", cmd)
    !{cmd}

print("Supervised done.")

In [ ]:
# ===================== SEMI-SUPERVISED (EmoSet, Section 4.2.1) =====================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if RUN_SSL:
    persistent_arg = f"--set persistent_dir={PERSISTENT_DIR}" if PERSISTENT_DIR else ""
    
    # 4000 labels, EMC-adaptive threshold
    print("=== SSL n=4000, rule=ratio (EMC-adaptive) ===")
    cmd = f"python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=4000 ssl.rule=ratio output_dir=/kaggle/working/runs/emoset_ssl_n4000_ratio {persistent_arg}"
    print("Running:", cmd)
    !{cmd}

    # 8000 labels, EMC-adaptive threshold
    print("=== SSL n=8000, rule=ratio (EMC-adaptive) ===")
    cmd = f"python -m ecc.ssl --config configs/ssl_fi.yaml --set ssl.n_labeled=8000 ssl.rule=ratio output_dir=/kaggle/working/runs/emoset_ssl_n8000_ratio {persistent_arg}"
    print("Running:", cmd)
    !{cmd}

print("SSL done.")

In [ ]:
# ===================== PREPARE RESULTS ZIP FOR ONE-CLICK DOWNLOAD =====================
import zipfile, os, glob, shutil

runs_dir = "/kaggle/working/runs"
zip_path = "/kaggle/working/emoset_results.zip"

run_dirs = sorted(glob.glob(os.path.join(runs_dir, "emoset_*")))

if not run_dirs:
    print("No emoset_* run directories found under", runs_dir)
else:
    stage_root = os.path.join(runs_dir, "_zip_stage")
    os.makedirs(stage_root, exist_ok=True)
    
    for run_dir in run_dirs:
        run_name = os.path.basename(run_dir)
        stage = os.path.join(stage_root, run_name)
        os.makedirs(stage, exist_ok=True)
        
        # best.pt -> best_model.pt
        best_pt = os.path.join(run_dir, "best.pt")
        results_json = os.path.join(run_dir, "results.json")
        history_jsonl = os.path.join(run_dir, "history.jsonl")
        config_json = os.path.join(run_dir, "config.json")
        
        if os.path.exists(best_pt):
            shutil.copy2(best_pt, os.path.join(stage, "best_model.pt"))
        if os.path.exists(results_json):
            shutil.copy2(results_json, os.path.join(stage, "results.json"))
        if os.path.exists(history_jsonl):
            shutil.copy2(history_jsonl, os.path.join(stage, "history.jsonl"))
        if os.path.exists(config_json):
            shutil.copy2(config_json, os.path.join(stage, "config.json"))
        
        print(f"Staged {run_name}")
    
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for stage in sorted(glob.glob(os.path.join(stage_root, "emoset_*"))):
            run_name = os.path.basename(stage)
            for fname in sorted(os.listdir(stage)):
                fpath = os.path.join(stage, fname)
                zf.write(fpath, arcname=os.path.join(run_name, fname))
    
    shutil.rmtree(stage_root, ignore_errors=True)
    
    print(f"\nCreated {zip_path}")
    print("Contents:")
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            print(f"  {info.filename} ({info.file_size:,} bytes)")
    
    print("\nDownload emoset_results.zip from the notebook output panel (right sidebar â†’ Output).")

In [ ]:
# ===================== AUTO-SAVE CHECKPOINT DATASET VERSION (OPTIONAL) =====================
# Requires Kaggle secrets: KAGGLE_USERNAME and KAGGLE_KEY (Settings -> Secrets)
import os
import subprocess

kaggle_user = os.environ.get("KAGGLE_USERNAME")
kaggle_key = os.environ.get("KAGGLE_KEY")

if not kaggle_user or not kaggle_key:
    print("\n=== Auto-save skipped: KAGGLE_USERNAME/KAGGLE_KEY not set ===")
    print("Add them in Kaggle Settings -> Secrets, then restart the session.")
    print("Or save manually: Dataset page -> Save new version")
else:
    import json as _json
    # Set up Kaggle API credentials
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as _f:
        _json.dump({"username": kaggle_user, "key": kaggle_key}, _f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    
    persistent = PERSISTENT_DIR if PERSISTENT_DIR else ""
    if persistent:
        parts = persistent.strip("/").split("/")
        if len(parts) >= 4:
            dataset_slug = f"{parts[2]}/{parts[3]}"
            print(f"Saving new version of checkpoint dataset: {dataset_slug}")
            result = subprocess.run(
                ["kaggle", "datasets", "version", "-p", persistent, "-m", "Auto-save checkpoint after training"],
                capture_output=True, text=True
            )
            print(result.stdout)
            
            if result.returncode != 0:
                print(f"Warning: Failed to save dataset version: {result.stderr}")
                print("You can save manually via Kaggle UI: Dataset -> Save new version")
        else:
            print(f"Could not parse dataset slug from PERSISTENT_DIR: {persistent}")
    else:
        print("PERSISTENT_DIR not set — skipping auto-save.")


## Resuming after a session timeout

Kaggle free sessions are capped at ~12 hours. Every epoch saves \checkpoint.pt\ to:
   1. The run's \output_dir\ (\/kaggle/working/runs/emoset_*\), and
   2. \PERSISTENT_DIR\ (your mounted Kaggle Dataset, if set).

If a session dies, re-run the same cell — it auto-detects \checkpoint.pt\ in the output dir
**and** in \PERSISTENT_DIR\, resuming from the last epoch.

To force a specific checkpoint, add \--set resume_from=/kaggle/working/runs/<run>/checkpoint.pt\.

## Saving persistent checkpoints between sessions

Before the session ends, save a new version of your checkpoint Dataset from the notebook
outputs (or via the Kaggle UI/API). Next session, mount the latest version.

## Download results

Run the cell above to create \emoset_results.zip\ containing, for each run:
- \est_model.pt\ — best-on-val model weights (from \est.pt\)
- esults.json\ — test metrics (ACC, ECC, EMC, ACC2)
- \history.jsonl\ — per-epoch training log
- \config.json\ — exact config used

Download \emoset_results.zip\ from the notebook output panel (right sidebar → Output) before
the session ends.